In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
data_dir = Path(r'C:\Users\DELL\Desktop\Pandas_in_14_days\Work\Data')
tables = {f.stem: pd.read_csv(f) for f in data_dir.glob('*.csv')}

for key, value in tables.items():
    print(key, value.shape)

olist_customers_dataset (99441, 5)
olist_geolocation_dataset (1000163, 5)
olist_orders_dataset (99441, 8)
olist_order_items_dataset (112650, 7)
olist_order_payments_dataset (103886, 5)
olist_order_reviews_dataset (99224, 7)
olist_products_dataset (32951, 9)
olist_sellers_dataset (3095, 4)
product_category_name_translation (71, 2)


In [3]:
customers = tables['olist_customers_dataset'].copy()
geolocation = tables['olist_geolocation_dataset'].copy()
orders = tables['olist_orders_dataset'].copy()
items = tables['olist_order_items_dataset'].copy()
payments = tables['olist_order_payments_dataset'].copy()
reviews = tables['olist_order_reviews_dataset'].copy()
products = tables['olist_products_dataset'].copy()
sellers = tables['olist_sellers_dataset'].copy()
category_translation = tables['product_category_name_translation'].copy()

df.groupby('column_to_group_by')['column_to_calculate'].aggregation_function()


Common Use Cases

• Single-Column Aggregation:
Group by one column and apply a summary function like sum(), mean(), count(), min(), or max().
#Total sales per category
df.groupby('category')['sales'].sum()

• Multi-Column Grouping:
Pass a list of columns to group by multiple criteria. Setting as_index=False keeps the grouped columns as regular DataFrame columns instead of turning them into a MultiIndex.

df.groupby(['region', 'category'], as_index=False)['sales'].mean()

• Multiple Aggregations at Once:
Use .agg() (or Named Aggregation) to compute several metrics simultaneously and rename the output columns cleanly.

df.groupby('category', as_index=False).agg(
    total_sales=('sales', 'sum'),
    avg_price=('price', 'mean'),
    order_count=('order_id', 'count')
)

• Group-Level Transformations:
Use .transform() when you want to return a result with the exact same shape and index as your original DataFrame—useful for calculating group percentages or filling missing values with a group average.

#Add a column showing each row's share of its category's total sales
df['pct_of_category'] = df['sales'] / df.groupby('category')['sales'].transform('sum')

• Filtering Groups:
Use .filter() to keep or drop entire groups based on a group-level condition.python
#Keep only categories with more than 100 total sales
df.groupby('category').filter(lambda x: x['sales'].sum() > 100)


### Helpful Parameters

• as_index=False. Mimics SQL-style GROUP BY output by preventing the grouping keys from becoming the DataFrame index.

• dropna=False. Includes NaN / null values as their own group instead of excluding them by default.

• sort=False. Disables automatic sorting of group keys to speed up performance on large datasets

## Task 1 - groupby with one and several keys, then agg with named aggregation

### 1.1 In payments, find the total payment_value for each payment_type, sorted from highest to lowest. Which type brings in the most money?

In [4]:
payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [5]:
payments.groupby('payment_type')['payment_value'].sum().sort_values(ascending=False)

payment_type
credit_card    12542084.19
boleto          2869361.27
voucher          379436.87
debit_card       217989.79
not_defined           0.00
Name: payment_value, dtype: float64

### 1.2 In items, for each product_id, calculate the number of rows, the total price and the average price in one agg call with named aggregation. Show the 5 products with the highest total.

In [6]:
items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [7]:
#Method 1
items.groupby('product_id', as_index=False).agg(
    num_of_rows = ('product_id','count'),
    total_price = ('price','sum'),
    avg_price = ('price','mean')
).sort_values(by='total_price', ascending=False).head()

,product_id,num_of_rows,total_price,avg_price
24086,bb50f2e236e5eea0100680137654686c,195,63885.00,327.615385
14068,6cdd53843498f92890544667809f1595,156,54730.20,350.834615
27613,d6160fb7873f184099d9bc95e30376af,35,48899.34,1397.124000
27039,d1c427060a0f73f6b889a5c7c61f2ac4,343,47214.51,137.651633
19742,99a4788cb24856965c36a24e339b6058,488,43025.56,88.167131


In [8]:
#Method 2
items.groupby('product_id').agg(
    n=('price', 'size'), 
    total=('price', 'sum'), 
    avg=('price', 'mean')).nlargest(5, 'total')

,n,total,avg
product_id,,,
bb50f2e236e5eea0100680137654686c,195,63885.00,327.615385
6cdd53843498f92890544667809f1595,156,54730.20,350.834615
d6160fb7873f184099d9bc95e30376af,35,48899.34,1397.124000
d1c427060a0f73f6b889a5c7c61f2ac4,343,47214.51,137.651633
99a4788cb24856965c36a24e339b6058,488,43025.56,88.167131


### 1.3 In customers, count customers for each (customer_state, customer_city) pair and show the top 10 pairs.

In [9]:
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [10]:
#Method 1
customers.groupby(['customer_state','customer_city'], as_index=False)['customer_unique_id'].size().nlargest(10,'size')

,customer_state,customer_city,size
4176,SP,sao paulo,15540
2788,RJ,rio de janeiro,6882
1062,MG,belo horizonte,2773
601,DF,brasilia,2131
2406,PR,curitiba,1521
3729,SP,campinas,1444
3208,RS,porto alegre,1379
372,BA,salvador,1245
3836,SP,guarulhos,1189
4160,SP,sao bernardo do campo,938


In [11]:
#Method 2
customers.groupby(['customer_state', 'customer_city']).size().sort_values(ascending=False).head(10)

customer_state  customer_city        
SP              sao paulo                15540
RJ              rio de janeiro            6882
MG              belo horizonte            2773
DF              brasilia                  2131
PR              curitiba                  1521
SP              campinas                  1444
RS              porto alegre              1379
BA              salvador                  1245
SP              guarulhos                 1189
                sao bernardo do campo      938
dtype: int64

### 1.4 In payments, calculate for each order_id the total payment_value and the number of payment rows. How many orders have more than one payment row?

In [12]:
payments.head(3)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71


In [13]:
result = payments.groupby('order_id').agg(
    total_value = ('payment_value','sum'),
    num_of_rows = ('order_id','size')
)
(result.num_of_rows > 1).sum()

np.int64(2961)

## Use value_counts (with normalize=True), nunique, size vs count

### 2.1 What percentage of orders has each order_status? Use value_counts(normalize=True) and round to 2 decimals. Which status is the largest after 'delivered'?

In [14]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [18]:
orders['order_status'].value_counts(normalize=True).mul(100).round(2)

order_status
delivered      97.02
shipped         1.11
canceled        0.63
unavailable     0.61
invoiced        0.32
processing      0.30
created         0.01
approved        0.00
Name: proportion, dtype: float64

### 2.2 In customers, compare nunique() of customer_id and customer_unique_id. What does the gap mean?

In [19]:
customers.head(3)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP


In [20]:
customers.customer_id.nunique()

99441

In [23]:
customers.customer_unique_id.nunique()

96096

In [22]:
customers.shape

(99441, 5)

customer_id is created for every order, while customer_unique_id identifies the person. The gap means some people ordered more than once.

### 2.3 In reviews, group by review_score and calculate both size() and count() of review_comment_message. Why are the numbers different?

In [28]:
reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [29]:
reviews.groupby('review_score').agg(
    size = ('review_comment_message','size'),
    count = ('review_comment_message','count')
)

,size,count
review_score,,
1,11424,8745
2,3151,2145
3,8179,3557
4,19142,5976
5,57328,20554


size() counts rows, and count() counts only non-null values. The difference is the number of reviews with no comment at each score.

## Task 3 - Use reset_index and know why you need it

### 3.1 Turn the result of Q1.1 into a DataFrame with columns payment_type and total_payment, using reset_index. Confirm the type is DataFrame.

In [35]:
payments.groupby('payment_type')['payment_value'].sum()

payment_type
boleto          2869361.27
credit_card    12542084.19
debit_card       217989.79
not_defined           0.00
voucher          379436.87
Name: payment_value, dtype: float64

In [33]:
df = payments.groupby('payment_type')['payment_value'].sum().reset_index(name='total_payment')

In [34]:
type(df)

pandas.DataFrame

### 3.2 Get the same table without reset_index, using as_index=False in the groupby. Do both give identical tables?

In [ ]:
payments.groupby('payment_type', a)['payment_value'].sum()